In [1]:
from fpdf import FPDF
from HARey.harey_main import HAReyMain
import random
from matplotlib.colors import to_hex

In [2]:
pdf = FPDF(orientation="L", unit="in", format="A4")

In [3]:
harey = HAReyMain()
harey.set_limiting_magnitude(5.5)

Loading constellations diagrams....     Done!
Loading star coordinates....     Done!
Computing stars colors...     Done!
Loading custom markers....       Done!
Loading the object names....       Done!


Using the tarot-round format, 2.75x4.75 in, using the template at /home/menegattig/Desktop/HARey/HARey/cardbacks/tarot_round.png


In [13]:
bookmarks = 45
IDS = ['Aql', 'Boo', 'Ori', 'Dra', 'And', 'Gem', 'Leo', 'Sgr', 'Sco', 'Tau']

description = {
    'Aql': "L'Aquila Reale", 
    'Boo': "Il Pastore dell'Orsa",
    'Ori': 'Il Cacciatore', 
    'Dra': 'Il Dragone',
    'And': 'La Principessa',
    'Gem': 'I Gemelli',
    'Leo': 'Il Leone', 
    'Sgr':  "L'Arciere",
    'Sco': 'Lo Scorpione',
    'Tau': 'Il Toro'
}

In [11]:
harey.set_card_template('temp')

harey.set_colors({'sky':"#1B2233", 'star':"#F4E6C4", 'constellations':'#C8C6B8', 'ecliptic':'#E66B6B', 'cardinal_markers': '#B94747'})

for id in IDS:
    harey.set_flags({'CON_LINES':True, 'SAVE':True, 'SHOW':False, 'ECLIPTIC':False})    
    harey.plot_card(id, save_name=f'book/{id}_lines.png', BEST_AR=True, star_size=400)
    harey.set_flags({'CON_LINES':False, 'SAVE':True, 'SHOW':False, 'ECLIPTIC':False})    
    harey.plot_card(id, save_name=f'book/{id}_bare.png', BEST_AR=True, star_size=400)

Using the temp format, 2.00x2.75 in, using the template at /home/menegattig/Desktop/HARey/HARey/cardbacks/tarot_round.png


In [12]:
def to_rgb(color):
    color = to_hex(color)
    return int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)

In [14]:
def filled_rounded_rect(pdf, x, y, w, h, r):
    """
    Draw a filled rounded rectangle without FPDF2 fill artifacts.
    
    pdf: FPDF object
    x, y: top-left corner
    w, h: width and height
    r: corner radius
    fill_color: (r,g,b) fill color
    border_color: (r,g,b) border color
    line_width: thickness of border
    """
    # Ensure radius is valid
    r = min(r, w/2, h/2)

    # --- Fill manually ---
    # 1. Central rectangle
    pdf.rect(x + 0.9*r, y + 0.9*r, w - 1.8*r, h - 1.8*r, style='F')
    
    # 2. Side rectangles
    pdf.rect(x + r, y, w - 2*r, r, style='F')       # top
    pdf.rect(x + r, y + h - r, w - 2*r, r, style='F')  # bottom
    pdf.rect(x, y + r, r, h - 2*r, style='F')       # left
    pdf.rect(x + w - r, y + r, r, h - 2*r, style='F')  # right
    
    # 3. Corner circles (using ellipse with equal width/height)
    pdf.ellipse(x, y, 2*r, 2*r, style='F')         # top-left
    pdf.ellipse(x + w -2*r, y, 2*r, 2*r, style='F')     # top-right
    pdf.ellipse(x, y + h -2*r, 2*r, 2*r, style='F')     # bottom-left
    pdf.ellipse(x + w -2*r, y + h -2*r, 2*r, 2*r, style='F') # bottom-right
    
    # --- Draw border on top ---
    pdf.rect(x, y, w, h, round_corners=True, corner_radius=r, style='D')


In [15]:
# Create landscape PDF in inches
from textwrap import fill


pdf = FPDF(orientation='L', unit='in', format='A4')

# Use page size from FPDF to avoid confusion
page_width = pdf.w    # 11.0 in (landscape)
page_height = pdf.h   # 8.5 in

box_width = 2
box_height = 6
boxes = 5

image_width = 2
image_height = 2.75
label_width = 1.5
label_height = box_height - 2*image_height

# Padding between and around boxes
pad_w = (page_width - boxes * box_width) / (boxes + 1)
pad_h = (page_height - box_height) / 2

# Font settings
pdf.add_font("The Walkyr", '', '/home/menegattig/Desktop/HARey/fonts/TheWalkyr.ttf')


pdf.set_auto_page_break(auto=False)

n_pages = (bookmarks + boxes - 1)//boxes

for i in range(n_pages):
    

    

    pdf.add_page()

    for i in range(boxes):

        id = random.sample(IDS, 1)[0]

        x_box = pad_w * (i + 1) + box_width * i 
        y_box = pad_h       
        
        pdf.image(f'book/{id}_bare.png', x_box, y_box, image_width, image_height)

        pdf.image(f'book/{id}_lines.png', x_box, y_box+image_height+label_height, image_width, image_height)


        pdf.set_fill_color(*to_rgb(harey.colors['sky']))
        pdf.set_draw_color(*to_rgb(harey.colors['sky']))
        pdf.rect(x_box, y_box + image_height, box_width, label_height, style='DF')

        round = 0.15

        pdf.set_line_width(0.02)
        pdf.set_fill_color(*to_rgb("#EAD17F"))
        pdf.set_draw_color(*to_rgb("#BFAF7F"))
        pdf.rect(x_box, y_box + image_height + label_height/3, box_width, label_height/3, style='DF')

        filled_rounded_rect(pdf, x_box + (image_width-label_width)/2, y_box + image_height, label_width, label_height, round)        

        pdf.set_text_color(*to_rgb(harey.colors['sky']))

        pdf.set_xy(x_box + (image_width-label_width)/2 +  round,  y_box + image_height + round/2)
        pdf.set_font("The Walkyr", size=12)
        pdf.cell(w=label_width- 2*round, h=label_height/2 - round/2, text=harey.names[id], border=0, align='C')
        pdf.set_xy(x_box + (image_width-label_width)/2 + round,  y_box + image_height + label_height/2)
        pdf.set_font("Times", 'I', size=11)
        pdf.cell(w=label_width -2*round, h=label_height/2 - round/2, text=description[id], border=0, align='C')

        pdf.rect(x_box, pad_h, box_width, box_height)


    pdf.add_page()

    for i in range(boxes):
        x_box = pad_w * (i + 1) + box_width * i 
        y_box = pad_h

        pdf.set_draw_color(0, 255, 255)
        pdf.rect(x_box, y_box, box_width, box_height)  
        with pdf.rotation(90, x=x_box + box_width/2, y=y_box + box_height/2):
            pdf.set_xy(x=x_box + box_width/2 - box_height/2, y=y_box + box_height/2 - box_width/2)
            pdf.multi_cell(w=box_height, h = box_width/2, text='HELLO FROM KERBIN \n HELLO DRES', border=1)

pdf.output('prova.pdf')